# Positional Encoding

Der Transformer liest alle Wörter auf einmal. Er hat keine Ahnung,
welches Wort zuerst kommt und welches zuletzt kommt. Aber die
Wortreihenfolge ist wichtig. Der Hund biss den Mann ist etwas ganz
anderes als Der Mann biss den Hund.

Deshalb müssen wir jedes Wort mit seiner Position versehen, bevor
wir es dem Modell zuführen. Der moderne Weg dafür heißt Rotary
Position Embeddings, kurz RoPE. Anstatt Positionszahlen zu addieren,
rotiert RoPE die Vektoren um einen Winkel, der davon abhängt, an
welcher Stelle das Wort im Satz steht.

Der raffinierte Teil dabei ist, dass nach der Rotation das
Skalarprodukt zwischen zwei beliebigen Wörtern nur noch von ihrem
Abstand zueinander abhängt. Nicht von ihren absoluten Positionen.
Wörter, die fünf Plätze auseinanderliegen, erhalten immer denselben
relativen Winkel, egal ob sie an Position 0 oder Position 500
stehen. Genau das sollte Attention interessieren.

Dieses Notebook zeigt dir die Rotationsmathematik und lässt dich
beobachten, wie sich ein Vektor verändert, während er sich durch
verschiedene Positionen bewegt.

## Importe

In [ ]:
import torch
import torch.nn as nn
import math

## RoPE-Implementierung

RoPE erwartet Eingabetensoren in der Form [batch, num_heads, seq_len, head_dim].
Dies ist dasselbe Layout, das auch innerhalb von Multi-Head-Attention verwendet wird.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even"

        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = freqs.repeat_interleave(2, dim=-1)

        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    @staticmethod
    def rotate_half(x):
        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]
        return torch.stack([-x_odd, x_even], dim=-1).flatten(-2)
    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## Rotation an einem einzelnen Vektor beobachten

In [ ]:
rope = RotaryPositionalEmbedding(d_model=4, max_seq_len=16)

q = torch.tensor([[[[0.8, 0.3, -0.5, 0.2]]]])  # [1, 1, 1, 4]
print(f"Original vector: {q[0, 0, 0].tolist()}")
print()

for pos in [0, 1, 2, 5]:
    rotated = rope(q, seq_len=pos + 1)
    last = rotated[0, 0, -1].tolist()
    vals = [f"{v:.3f}" for v in last]
    print(f"Position {pos}: [{', '.join(vals)}]")

## Eigenschaft der relativen Position

Das Skalarprodukt zwischen zwei rotierten Vektoren hängt nur von
ihrem Abstand zueinander ab. Nicht von ihren absoluten Positionen.

In [ ]:
rope = RotaryPositionalEmbedding(d_model=64, max_seq_len=256)

seq_len = 12
num_heads = 4
head_dim = 64

q = torch.randn(1, num_heads, seq_len, head_dim)
k = torch.randn(1, num_heads, seq_len, head_dim)

q_rot = rope(q, seq_len=seq_len)
k_rot = rope(k, seq_len=seq_len)

head = 0
a = (q_rot[0, head, 2] @ k_rot[0, head, 4]).item()
b = (q_rot[0, head, 5] @ k_rot[0, head, 7]).item()
c = (q_rot[0, head, 0] @ k_rot[0, head, 8]).item()

print(f"Distance 2 (positions 2,4):     {a:.4f}")
print(f"Distance 2 (positions 5,7):     {b:.4f}")
print(f"Distance 8 (positions 0,8):     {c:.4f}")
print()
print("Same distance gives similar score regardless of position.")

## RoPE innerhalb von Attention

RoPE wird nur auf Queries und Keys angewendet. Values bleiben
unverändert. Dadurch wird der Attention-Score positionsbewusst,
während der Inhalt sauber bleibt.

In [ ]:
head_dim = 64
num_heads = 4
seq_len = 10

rope = RotaryPositionalEmbedding(d_model=head_dim, max_seq_len=128)

q = torch.randn(1, num_heads, seq_len, head_dim)
k = torch.randn(1, num_heads, seq_len, head_dim)

q_rot = rope(q, seq_len)
k_rot = rope(k, seq_len)

scores = (q_rot @ k_rot.transpose(-2, -1)) / math.sqrt(head_dim)
print(f"Attention scores shape: {scores.shape}")
print(f"Token 0 attends to itself:  {scores[0, 0, 0, 0]:.4f}")
print(f"Token 0 attends to token 5: {scores[0, 0, 0, 5]:.4f}")